# 157 — Position Encoding Analysis

Position embeddings tell the model where each token is. This module analyzes
the structure of position embeddings, how positional information persists
through layers, and how much model capacity positions consume.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.position_encoding_analysis import (
    position_embedding_structure,
    position_content_separation,
    position_info_persistence,
    position_attention_pattern,
    position_encoding_capacity,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

## Position Embedding Structure

What does the position embedding matrix look like? How many effective
dimensions does it use?

In [ ]:
result = position_embedding_structure(model)
for k, v in result.items():
    print(f"{k}: {v}")

## Position vs Content Separation

How well are position and content information separated in the residual?

In [ ]:
result = position_content_separation(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: pos_fraction={p['position_fraction']:.4f}  "
          f"total_var={p['total_variance']:.6f}")

## Position Information Persistence

Can position still be decoded from the residual at later layers?

In [ ]:
result = position_info_persistence(model, tokens)
for p in result['per_layer']:
    bar = '#' * int(abs(p['mean_position_cosine']) * 30)
    print(f"Layer {p['layer']}: pos_cosine={p['mean_position_cosine']:+.4f}  {bar}")

## Position-Dependent Attention

How much do attention patterns depend on relative position vs content?

In [ ]:
result = position_attention_pattern(model, tokens)
for h in result['per_head']:
    bar = '#' * int(h['positional_bias'] * 30)
    print(f"L{h['layer']}H{h['head']}: pos_bias={h['positional_bias']:.4f}  {bar}")

## Position Encoding Capacity

How many d_model dimensions do position embeddings consume?

In [ ]:
result = position_encoding_capacity(model)
print(f"Dims for 90% variance: {result['dims_for_90pct']}/{result['d_model']} "
      f"({result['capacity_fraction_90']:.0%})")
print(f"Dims for 95%: {result['dims_for_95pct']}")
print(f"Dims for 99%: {result['dims_for_99pct']}")
print(f"Position/token norm ratio: {result['pos_to_token_ratio']:.3f}")